# 连通性与字段名核对

连接配置从 `vol_strategy.py` 读，改那里的 `MODE` 就行，这里不重复配置。

- `MODE = "dma"` — 直连 Rotman 服务器，Mac/任意系统可用，**不需要 RIT Client**
- `MODE = "client"` — 连本机 Windows RIT Client 的 `localhost:9999`，需要桌面版 Client 已启动并登录

注意：浏览器版 RIT 和 Mac app 都**不会**在本机开 `localhost:9999`，它们走的是 DMA。

In [3]:
import requests
import vol_strategy as vs

session = requests.Session()
session.headers.update(vs.AUTHORIZATION)
print("MODE:", vs.MODE)
print("endpoint:", vs.API_ENDPOINT)

MODE: dma
endpoint: http://flserver.rotman.utoronto.ca:16595/v1


In [4]:
# 1. 连通性诊断。不要用 resp.json()，401 的返回体可能不是 JSON。
try:
    resp = session.get(f"{vs.API_ENDPOINT}/case", timeout=8)
except Exception as e:
    print("连不上：", type(e).__name__)
    print("-> MODE=client 时：Windows 桌面版 RIT Client 没启动/没登录")
    print("-> 浏览器版和 Mac app 不提供 localhost:9999，请改用 MODE=\"dma\"")
else:
    print("HTTP", resp.status_code, resp.reason)
    print(resp.text[:400] or "(空)")
    if resp.status_code == 401:
        print("\n401 -> 凭证不对，或 Client 未登录到案例")

HTTP 200 OK
{"name":"RITCx Volatility Trading Case","period":1,"tick":110,"ticks_per_period":300,"total_periods":1,"status":"ACTIVE","is_enforce_trading_limits":true}


In [5]:
# 2. 第一次拉新闻，不带游标
resp = session.get(f"{vs.API_ENDPOINT}/news", params={"limit": 20})
print(resp.status_code)
news = resp.json()
news

200


[{'news_id': 4,
  'period': 1,
  'tick': 112,
  'ticker': None,
  'headline': 'News 2',
  'body': 'The analysts have informed you that the realized volatility of RTM next week will be between 16% and 21%'},
 {'news_id': 3,
  'period': 1,
  'tick': 75,
  'ticker': 'Week 2',
  'headline': 'Announcement 1',
  'body': 'The analysts have informed you that the realized volatility of RTM this week will be 31%'},
 {'news_id': 2,
  'period': 1,
  'tick': 36,
  'ticker': None,
  'headline': 'News 1',
  'body': 'The analysts have informed you that the realized volatility of RTM next week will be between 31% and 36%'},
 {'news_id': 1,
  'period': 1,
  'tick': 1,
  'ticker': 'Week 1',
  'headline': 'Risk free rate and current annualized volatility of RTM',
  'body': 'The current risk free rate is 0%. RTM is an ETF that mimics one of the major indices in the simulated world and its current annualized realized volatility is 19%. This simulation consists of 20 trading days that are each 15 ticks in le

In [6]:
# 3. 核对字段名是否为 news_id / period / tick / ticker / headline / body
if news:
    print(sorted(news[0].keys()))
    print(news[0])

['body', 'headline', 'news_id', 'period', 'tick', 'ticker']
{'news_id': 4, 'period': 1, 'tick': 112, 'ticker': None, 'headline': 'News 2', 'body': 'The analysts have informed you that the realized volatility of RTM next week will be between 16% and 21%'}


In [7]:
# 4. 增量拉取：只要 news_id 比游标大的
last_news_id = max(n["news_id"] for n in news) if news else 0
print("last_news_id =", last_news_id)

resp2 = session.get(f"{vs.API_ENDPOINT}/news", params={"after": last_news_id, "limit": 20})
print(resp2.status_code)
resp2.json()

last_news_id = 4
200


[{'news_id': 5,
  'period': 1,
  'tick': 150,
  'ticker': 'Week 3',
  'headline': 'Announcement 2',
  'body': 'The analysts have informed you that the realized volatility of RTM this week will be 17%'},
 {'news_id': 4,
  'period': 1,
  'tick': 112,
  'ticker': None,
  'headline': 'News 2',
  'body': 'The analysts have informed you that the realized volatility of RTM next week will be between 16% and 21%'}]

In [8]:
# 5. 用真实公告文本验证波动率解析
for n in sorted(news, key=lambda x: x["news_id"]):
    print(f"tick={n['tick']:>3}  {str(vs.parse_vol_from_news(n)):26}  {n['body'][:60]}")

state = vs.new_vol_state()
vs.apply_news_to_state(news, state)
print("\nstate ->", state)

tick=  1  ('this', 0.19)              The current risk free rate is 0%. RTM is an ETF that mimics 
tick= 36  ('next', (0.31, 0.36))      The analysts have informed you that the realized volatility 
tick= 75  ('this', 0.31)              The analysts have informed you that the realized volatility 
tick=112  ('next', (0.16, 0.21))      The analysts have informed you that the realized volatility 

state -> {'current_vol': 0.31, 'next_range': (0.16, 0.21), 'last_news_id': 0}


In [ ]:
# 6. 核对 /securities 的字段名（build_signal_table 依赖 ticker/last/bid/ask/position）
sec = session.get(f"{vs.API_ENDPOINT}/securities").json()
print(sorted(sec[0].keys()))
sec[:3]